In [59]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
import pandas as pd
import numpy as np

train = pd.read_csv("/content/drive/MyDrive/house-prices/train.csv")
test = pd.read_csv("/content/drive/MyDrive/house-prices/test.csv")

print(train.shape, test.shape)
train.head()

(1460, 81) (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [61]:
train["SalePrice"].describe()

# أكتر 20 عمود فيهم قيم ناقصة
train.isnull().sum().sort_values(ascending=False).head(20)

,0
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageQual,81
GarageFinish,81
GarageType,81


In [62]:
train["MasVnrType"].value_counts(dropna=False)
train[train["MasVnrType"].isnull() & train["MasVnrArea"].notnull()].shape

(864, 81)

In [63]:
# هل البيوت اللي مفيهاش جراج مساحة جراجها صفر؟
no_garage = train["GarageType"].isnull()
print(train.loc[no_garage, "GarageArea"].value_counts())

# نفس الفكرة للبدروم
no_bsmt = train["BsmtQual"].isnull()
print(train.loc[no_bsmt, "TotalBsmtSF"].value_counts())

GarageArea
0    81
Name: count, dtype: int64
TotalBsmtSF
0    37
Name: count, dtype: int64


In [64]:
test.isnull().sum().sort_values(ascending=False).head(35)

,0
PoolQC,1456
MiscFeature,1408
Alley,1352
Fence,1169
MasVnrType,894
FireplaceQu,730
LotFrontage,227
GarageYrBlt,78
GarageCond,78
GarageFinish,78


In [65]:
cols = ["GarageType", "GarageYrBlt", "GarageFinish", "GarageCars", "GarageArea"]
test.loc[test["GarageType"].notnull() & test["GarageYrBlt"].isnull(), cols]

,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea
666,Detchd,NaN,NaN,1.0,360.0
1116,Detchd,NaN,NaN,NaN,NaN


In [66]:
det = train[train["GarageType"] == "Detchd"]
print(det["GarageFinish"].value_counts())
print(det["GarageArea"].median(), det["GarageCars"].median())

GarageFinish
Unf    354
RFn     19
Fin     14
Name: count, dtype: int64
400.0 2.0


In [67]:
idx = [666, 1116]
test.loc[idx, "GarageFinish"] = "Unf"
test.loc[idx, "GarageYrBlt"] = test.loc[idx, "YearBuilt"]
test.loc[1116, "GarageCars"] = 2.0
test.loc[1116, "GarageArea"] = 400.0

test.loc[idx, ["GarageType", "GarageYrBlt", "GarageFinish", "GarageCars", "GarageArea"]]

,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea
666,Detchd,1910.0,Unf,1.0,360.0
1116,Detchd,1923.0,Unf,2.0,400.0


In [68]:
bsmt = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF"]

text_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

test.loc[test[text_cols].isnull().any(axis=1) & (test["TotalBsmtSF"] != 0), bsmt]

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF
27,Gd,TA,NaN,Unf,Unf,0.0,0.0,1595.0,1595.0
580,Gd,NaN,Mn,GLQ,Rec,1044.0,382.0,0.0,1426.0
660,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
725,TA,NaN,No,BLQ,Unf,1033.0,0.0,94.0,1127.0
757,NaN,Fa,No,Unf,Unf,0.0,0.0,173.0,173.0
758,NaN,TA,No,Unf,Unf,0.0,0.0,356.0,356.0
888,Gd,TA,NaN,Unf,Unf,0.0,0.0,725.0,725.0
1064,TA,NaN,Av,ALQ,Unf,755.0,0.0,240.0,995.0


In [69]:
for col in ["BsmtExposure", "BsmtCond", "BsmtQual"]:
    print(train[col].value_counts())
    print("---")

BsmtExposure
No    953
Av    221
Gd    134
Mn    114
Name: count, dtype: int64
---
BsmtCond
TA    1311
Gd      65
Fa      45
Po       2
Name: count, dtype: int64
---
BsmtQual
TA    649
Gd    618
Ex    121
Fa     35
Name: count, dtype: int64
---


In [70]:
for col in ["BsmtExposure", "BsmtCond", "BsmtQual"]:
    mode = train[col].mode()[0]
    mask = test[col].isnull() & (test["TotalBsmtSF"] > 0)
    test.loc[mask, col] = mode

test.loc[[27, 580, 725, 757, 758, 888, 1064], ["BsmtQual", "BsmtCond", "BsmtExposure"]]

,BsmtQual,BsmtCond,BsmtExposure
27,Gd,TA,No
580,Gd,TA,Mn
725,TA,TA,No
757,TA,Fa,No
758,TA,TA,No
888,Gd,TA,No
1064,TA,TA,Av


In [71]:
text_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
num_cols = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"]

test.loc[660, text_cols] = "None"
test.loc[660, num_cols] = 0

test.loc[[660], text_cols + num_cols]

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,BsmtFullBath,BsmtHalfBath
660,None,None,None,None,None,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
             "GarageType", "GarageFinish", "GarageQual", "GarageCond",
             "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

for col in none_cols:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

print(train[none_cols].isnull().sum().sum())
print(test[none_cols].isnull().sum().sum())

0
0


In [73]:
cols = ["MasVnrType", "MasVnrArea"]

print(train[train["MasVnrType"].isnull() & (train["MasVnrArea"] != 0)][cols])
print(test[test["MasVnrType"].isnull() & (test["MasVnrArea"] != 0)][cols])

     MasVnrType  MasVnrArea
234         NaN         NaN
529         NaN         NaN
624         NaN       288.0
650         NaN         NaN
773         NaN         1.0
936         NaN         NaN
973         NaN         NaN
977         NaN         NaN
1230        NaN         1.0
1243        NaN         NaN
1278        NaN         NaN
1300        NaN       344.0
1334        NaN       312.0
     MasVnrType  MasVnrArea
209         NaN       285.0
231         NaN         NaN
246         NaN         NaN
422         NaN         NaN
532         NaN         NaN
544         NaN         NaN
581         NaN         NaN
851         NaN         NaN
865         NaN         NaN
880         NaN         NaN
889         NaN         NaN
908         NaN         NaN
992         NaN         1.0
1132        NaN         NaN
1150        NaN       198.0
1197        NaN         NaN
1226        NaN         NaN
1402        NaN         NaN


In [74]:
print(train["MasVnrType"].value_counts())

MasVnrType
BrkFace    445
Stone      128
BrkCmn      15
Name: count, dtype: int64


In [75]:
print(train.groupby("MasVnrType")["MasVnrArea"].describe())

            count        mean         std   min    25%    50%     75%     max
MasVnrType                                                                   
BrkCmn       15.0  247.666667  182.423239  41.0   91.5  192.0  381.50   621.0
BrkFace     445.0  259.008989  215.137888   0.0  114.0  202.0  336.00  1600.0
Stone       128.0  239.304688  172.129257   0.0  111.5  206.5  296.25   860.0


In [76]:
for df in [train, test]:
    real = df["MasVnrType"].isnull() & (df["MasVnrArea"] > 1)
    df.loc[real, "MasVnrType"] = "BrkFace"
    df["MasVnrType"] = df["MasVnrType"].fillna("None")
    df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
    df.loc[df["MasVnrType"] == "None", "MasVnrArea"] = 0

print(train["MasVnrType"].value_counts())
print(train[["MasVnrType", "MasVnrArea"]].isnull().sum().sum())
print(test[["MasVnrType", "MasVnrArea"]].isnull().sum().sum())

MasVnrType
None       869
BrkFace    448
Stone      128
BrkCmn      15
Name: count, dtype: int64
0
0


In [77]:
for name, df in [("train", train), ("test", test)]:
    miss = df.isnull().sum()
    print(name)
    print(miss[miss > 0])
    print("---")

train
LotFrontage    259
Electrical       1
GarageYrBlt     81
dtype: int64
---
test
MSZoning          4
LotFrontage     227
Utilities         2
Exterior1st       1
Exterior2nd       1
BsmtFullBath      1
BsmtHalfBath      1
KitchenQual       1
Functional        2
GarageYrBlt      76
SaleType          1
dtype: int64
---


In [78]:
for name, df in [("train", train), ("test", test)]:
    no_year = df["GarageYrBlt"].isnull()
    no_garage = df["GarageType"] == "None"
    print(name, no_year.sum(), no_garage.sum(), (no_year & no_garage).sum())

train 81 81 81
test 76 76 76


In [79]:
for df in [train, test]:
    df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

print(train["GarageYrBlt"].isnull().sum())
print(test["GarageYrBlt"].isnull().sum())
print((train["GarageYrBlt"] == 0).sum(), (test["GarageYrBlt"] == 0).sum())

0
0
81 76


In [80]:
print(train["Electrical"].value_counts())

mode = train["Electrical"].mode()[0]
train["Electrical"] = train["Electrical"].fillna(mode)

print(mode)
print(train["Electrical"].isnull().sum())

Electrical
SBrkr    1334
FuseA      94
FuseF      27
FuseP       3
Mix         1
Name: count, dtype: int64
SBrkr
0


In [81]:
print(train.groupby("Neighborhood")["LotFrontage"].median().sort_values())
print(train["LotFrontage"].median())

Neighborhood
BrDale     21.0
MeadowV    21.0
NPkVill    24.0
Blueste    24.0
Blmngtn    43.0
BrkSide    52.0
IDOTRR     60.0
OldTown    60.0
SWISU      60.0
StoneBr    61.5
Gilbert    65.0
Edwards    65.5
SawyerW    66.5
Veenker    68.0
CollgCr    70.0
Sawyer     71.0
NAmes      73.0
Mitchel    73.0
Somerst    73.5
Crawfor    74.0
ClearCr    80.0
NWAmes     80.0
Timber     85.0
NridgHt    88.5
NoRidge    91.0
Name: LotFrontage, dtype: float64
69.0


In [82]:
medians = train.groupby("Neighborhood")["LotFrontage"].median()
global_median = train["LotFrontage"].median()

for df in [train, test]:
    fill = df["Neighborhood"].map(medians).fillna(global_median)
    df["LotFrontage"] = df["LotFrontage"].fillna(fill)

print(train["LotFrontage"].isnull().sum())
print(test["LotFrontage"].isnull().sum())

0
0


In [83]:
cat_cols = ["MSZoning", "Utilities", "Exterior1st", "Exterior2nd",
            "KitchenQual", "Functional", "SaleType"]

for col in cat_cols:
    mode = train[col].mode()[0]
    test[col] = test[col].fillna(mode)
    print(col, mode)

miss = test.isnull().sum()
print(miss[miss > 0])

MSZoning RL
Utilities AllPub
Exterior1st VinylSd
Exterior2nd VinylSd
KitchenQual TA
Functional Typ
SaleType WD
BsmtFullBath    1
BsmtHalfBath    1
dtype: int64


In [84]:
cols = ["BsmtQual", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"]
print(test.loc[test["BsmtFullBath"].isnull() | test["BsmtHalfBath"].isnull(), cols])

    BsmtQual  TotalBsmtSF  BsmtFullBath  BsmtHalfBath
728     None          0.0           NaN           NaN


In [85]:
test.loc[728, ["BsmtFullBath", "BsmtHalfBath"]] = 0

print(train.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [86]:
import numpy as np

price = train["SalePrice"]
print("الالتواء قبل:", price.skew())
print("الالتواء بعد:", np.log1p(price).skew())

الالتواء قبل: 1.8828757597682129
الالتواء بعد: 0.12134661989685333


In [87]:
y = np.log1p(train["SalePrice"])
X = train.drop(columns=["SalePrice", "Id"])

test_ids = test["Id"]
X_test = test.drop(columns=["Id"])

print(X.shape, X_test.shape)
print(X.dtypes.value_counts())

(1460, 79) (1459, 79)
object     43
int64      33
float64     3
Name: count, dtype: int64


In [88]:
print(X["MSSubClass"].value_counts().sort_index())

MSSubClass
20     536
30      69
40       4
45      12
50     144
60     299
70      60
75      16
80      58
85      20
90      52
120     87
160     63
180     10
190     30
Name: count, dtype: int64


In [89]:
X["MSSubClass"] = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

print(X["MSSubClass"].dtype, X_test["MSSubClass"].dtype)
print(X.dtypes.value_counts())

object object
object     44
int64      32
float64     3
Name: count, dtype: int64


In [90]:
all_data = pd.concat([X, X_test], axis=0, keys=["train", "test"])
all_enc = pd.get_dummies(all_data)

X_enc = all_enc.loc["train"]
X_test_enc = all_enc.loc["test"]

print(X_enc.shape, X_test_enc.shape)
print((X_enc.dtypes == "object").sum())
print(list(X_enc.columns) == list(X_test_enc.columns))

(1460, 317) (1459, 317)
0
True


In [91]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

scores = cross_val_score(model, X_enc, y, cv=5, scoring="neg_root_mean_squared_error")
rmse = -scores

print(rmse)
print("المتوسط:", rmse.mean())

[0.13875933 0.15328798 0.14386233 0.1291564  0.14892377]
المتوسط: 0.14279796167121966


In [92]:
final_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
final_model.fit(X_enc, y)

pred_log = final_model.predict(X_test_enc)
pred_price = np.expm1(pred_log)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": pred_price})
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(submission["SalePrice"].describe())

     Id      SalePrice
0  1461  125974.052500
1  1462  155257.848045
2  1463  175933.805528
3  1464  181509.468968
4  1465  197222.400824
count      1459.000000
mean     176449.851528
std       72171.341350
min       55414.634174
25%      128905.318024
50%      156583.510311
75%      207586.584939
max      490263.935239
Name: SalePrice, dtype: float64


In [93]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [94]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=3, random_state=42)

scores = cross_val_score(gb, X_enc, y, cv=5, scoring="neg_root_mean_squared_error")
rmse = -scores

print(rmse)
print("المتوسط:", rmse.mean())

[0.11389319 0.14062844 0.12884024 0.11327813 0.12035439]
المتوسط: 0.12339887685252982


In [95]:
final_gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                     max_depth=3, random_state=42)
final_gb.fit(X_enc, y)

pred_price = np.expm1(final_gb.predict(X_test_enc))

sub_gb = pd.DataFrame({"Id": test_ids, "SalePrice": pred_price})
sub_gb.to_csv("submission_gb.csv", index=False)

print(sub_gb["SalePrice"].describe())
files.download("submission_gb.csv")

count      1459.000000
mean     177247.968916
std       76127.979578
min       34424.611407
25%      127234.294576
50%      156922.365367
75%      207511.761481
max      561890.561602
Name: SalePrice, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [96]:
cols = ["GrLivArea", "OverallQual", "SalePrice"]
print(train.loc[train["GrLivArea"] > 4000, cols])

      GrLivArea  OverallQual  SalePrice
523        4676           10     184750
691        4316           10     755000
1182       4476           10     745000
1298       5642           10     160000


In [97]:
drop_idx = [523, 1298]
X_clean = X_enc.drop(index=drop_idx)
y_clean = y.drop(index=drop_idx)

print(X_clean.shape, y_clean.shape)

scores = cross_val_score(gb, X_clean, y_clean, cv=5, scoring="neg_root_mean_squared_error")
rmse = -scores

print(rmse)
print("المتوسط:", rmse.mean())

(1458, 317) (1458,)
[0.1130263  0.12485565 0.12718595 0.1087024  0.11775691]
المتوسط: 0.11830544185987732


In [98]:
final_gb2 = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                      max_depth=3, random_state=42)
final_gb2.fit(X_clean, y_clean)

pred_price = np.expm1(final_gb2.predict(X_test_enc))

sub_gb2 = pd.DataFrame({"Id": test_ids, "SalePrice": pred_price})
sub_gb2.to_csv("submission_gb_clean.csv", index=False)

print(sub_gb2["SalePrice"].describe())
files.download("submission_gb_clean.csv")

count      1459.000000
mean     178204.216350
std       79560.030641
min       34505.922649
25%      127996.688372
50%      156338.500018
75%      208105.594237
max      765534.194906
Name: SalePrice, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [99]:
big = test["GrLivArea"] > 4000

out = pd.DataFrame({
    "Id": test.loc[big, "Id"],
    "GrLivArea": test.loc[big, "GrLivArea"],
    "OverallQual": test.loc[big, "OverallQual"],
    "before_removal": sub_gb.loc[big, "SalePrice"],
    "after_removal": sub_gb2.loc[big, "SalePrice"],
})
print(out)

        Id  GrLivArea  OverallQual  before_removal  after_removal
1089  2550       5095           10    268645.26128  765534.194906


In [100]:
import xgboost
print(xgboost.__version__)

3.4.1


In [101]:
print([name for name in dir() if name.startswith(("X", "y"))])

['X', 'XGBRegressor', 'X_clean', 'X_enc', 'X_test', 'X_test_enc', 'y', 'y_clean']


In [102]:
print(X_enc.shape, y.shape)
print(y.head(3))

(1460, 317) (1460,)
0    12.247699
1    12.109016
2    12.317171
Name: SalePrice, dtype: float64


In [103]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42)

for name, model in [("XGBoost", xgb), ("GradientBoosting", gb)]:
    scores = -cross_val_score(model, X_enc, y, cv=kf, scoring="neg_root_mean_squared_error")
    print(name, scores.round(4), "mean =", scores.mean().round(4))

XGBoost [0.1318 0.1119 0.1689 0.1249 0.1104] mean = 0.1296
GradientBoosting [0.136  0.1102 0.1681 0.1292 0.1102] mean = 0.1308


In [104]:
final_xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42)
final_xgb.fit(X_enc, y)

pred_price = np.expm1(final_xgb.predict(X_test_enc))

sub_xgb = pd.DataFrame({"Id": test_ids, "SalePrice": pred_price})
sub_xgb.to_csv("submission_xgb.csv", index=False)

print(sub_xgb["SalePrice"].describe())
files.download("submission_xgb.csv")

count      1459.000000
mean     177212.140625
std       76018.015625
min       36765.468750
25%      127879.320312
50%      156988.015625
75%      207143.093750
max      538160.125000
Name: SalePrice, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>